# Bangladesh Informal Firms Analysis - Part 2
## Continuing from Part 1: Business Size Analysis and Financing Sources

In [ ]:
# Re-import libraries and reload data (assuming this runs after Part 1)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from matplotlib.ticker import FuncFormatter
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Reload data with ISIC classifications
surveydata = pd.read_csv('informality_data_NEW.csv')
isicdata = pd.read_csv('ISIC_Rev_3_english_structure.csv')

# Recreate ISIC mappings
surveydata['q1_2'] = surveydata['q1_2'].astype(str)
surveydata['ISIC2_code'] = surveydata['q1_2'].str[:2]

isic_section_map = dict(zip(isicdata['Code'].astype(str), isicdata['SectionCode']))
isic_section_desc_map = dict(zip(isicdata['Code'].astype(str), isicdata['SectionDesc']))
isic4_desc_map = dict(zip(isicdata['Code'].astype(str), isicdata['Description']))

surveydata['ISIC_section'] = surveydata['q1_2'].map(isic_section_map)
surveydata['ISIC_sectiondesc'] = surveydata['q1_2'].map(isic_section_desc_map)
surveydata['ISIC4_desc'] = surveydata['q1_2'].map(isic4_desc_map)

# Format function
def format_thousands(x, pos):
    return f'{x:,.0f}'
formatter = FuncFormatter(format_thousands)

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)

## Exploring Business Size - Revenue vs Number of Employees

As another measure of business size, how does the number of employees vary across ISIC sections?

In [ ]:
# Box plot of number of employees by ISIC section
plt.figure(figsize=(15, 10))

# Filter data
emp_data = surveydata.dropna(subset=['ISIC_sectiondesc', 'q1_12a5a'])
emp_data = emp_data[emp_data['ISIC_sectiondesc'] != '']

# Create box plot
sns.boxplot(data=emp_data, y='ISIC_sectiondesc', x='q1_12a5a', 
           palette='Set3', orient='h')

plt.xlim(0, 50)
plt.xlabel('Number of Employees', fontsize=12)
plt.ylabel('ISIC Section', fontsize=12)
plt.title('Number of Employees by ISIC Section', fontsize=16, fontweight='bold')

# Wrap y-axis labels
labels = [label.get_text() for label in plt.gca().get_yticklabels()]
wrapped_labels = ['\n'.join([label[i:i+20] for i in range(0, len(label), 20)]) for label in labels]
plt.gca().set_yticklabels(wrapped_labels, fontsize=10)

plt.tight_layout()
plt.show()

Based on this boxplot, section Hotels & Restaurants has the highest median number of employees, followed by Manufacturing. The medians for those sections are significantly higher than the other sections, and the difference between 1st and 3rd quartiles wider.

### Point Plot comparison - Annual Revenue & Number of Employees by ISIC section
Now, let's explore the two size characteristics together with the ISIC section, all on the same point plot.

In [ ]:
# Scatter plots with ISIC sections colored
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Prepare data
scatter_data = surveydata.dropna(subset=['q4_7', 'q1_12a5a', 'ISIC_sectiondesc'])
scatter_data = scatter_data[scatter_data['ISIC_sectiondesc'] != '']

# Plot 1: Regular scale
for i, section in enumerate(scatter_data['ISIC_sectiondesc'].unique()):
    section_data = scatter_data[scatter_data['ISIC_sectiondesc'] == section]
    ax1.scatter(section_data['q1_12a5a'], section_data['q4_7'], 
               label=section, alpha=0.6, s=30, color=plt.cm.tab10(i))

ax1.set_xlim(0, 50)
ax1.set_ylim(0, 10000000)
ax1.set_xlabel('Number of Employees')
ax1.set_ylabel('Annual Revenue (Takas)')
ax1.set_title('Revenue vs Employees by ISIC Section')
ax1.yaxis.set_major_formatter(formatter)
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

# Plot 2: Log scale with jitter
for i, section in enumerate(scatter_data['ISIC_sectiondesc'].unique()):
    section_data = scatter_data[scatter_data['ISIC_sectiondesc'] == section]
    # Remove zeros for log transformation
    section_data_pos = section_data[section_data['q4_7'] > 0]
    if len(section_data_pos) > 0:
        # Add jitter to x-axis
        x_jitter = section_data_pos['q1_12a5a'] + np.random.normal(0, 0.1, len(section_data_pos))
        ax2.scatter(x_jitter, np.log10(section_data_pos['q4_7']), 
                   label=section, alpha=0.3, s=20, color=plt.cm.tab10(i))

ax2.set_xlim(0, 20)
ax2.set_xlabel('Number of Employees (with jitter)')
ax2.set_ylabel('Log10(Annual Revenue)')
ax2.set_title('Log Scale with Jitter by ISIC Section')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

What is apparent in the first plot is made more explicit in the second plot (the one with the logarithmic adjustment applied to annual revenue): how intensively different ISIC sections utilise labor. ISIC section Retail & Repair has revenue that grows very quickly for a small growth of number of employees; however, in sectors Hotels & Restaurants and Manufacturing, annual revenue increases more slowly as number of employees grow--implying that those sectors are more dependent on labor for growth.

Let's explore the correlation between revenue and number of employees by section.

In [ ]:
# Calculate correlations by ISIC section
sections_to_analyze = ['Manufacturing', 'Retail & Repair', 'Hotels and restaurants', 'Other Community']
section_codes = {'Manufacturing': 'D', 'Retail & Repair': 'G', 
                'Hotels and restaurants': 'H', 'Other Community': 'O'}

correlation_results = {}

print("=== Correlation Analysis: Annual Revenue vs Number of Employees by ISIC Section ===")
print()

for section_name in sections_to_analyze:
    section_code = section_codes.get(section_name)
    if section_code:
        section_data = surveydata[
            (surveydata['ISIC_section'] == section_code) & 
            (surveydata['q4_7'].notna()) & 
            (surveydata['q1_12a5a'].notna())
        ]
        
        if len(section_data) > 10:  # Minimum sample size
            corr_coeff, p_value = stats.pearsonr(section_data['q4_7'], section_data['q1_12a5a'])
            correlation_results[section_name] = {'correlation': corr_coeff, 'p_value': p_value, 'n': len(section_data)}
            
            print(f"**{section_name}** (ISIC Section {section_code})")
            print(f"  Correlation coefficient: {corr_coeff:.3f}")
            print(f"  P-value: {p_value:.6f}")
            print(f"  Sample size: {len(section_data)}")
            print(f"  Significance: {'Yes' if p_value < 0.05 else 'No'} (p < 0.05)")
            print()

# Create visualization of correlations
if correlation_results:
    sections = list(correlation_results.keys())
    correlations = [correlation_results[s]['correlation'] for s in sections]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(sections, correlations, 
                  color=['red' if correlation_results[s]['p_value'] >= 0.05 else 'blue' for s in sections],
                  alpha=0.7)
    
    plt.title('Correlation between Revenue and Number of Employees by ISIC Section', 
             fontsize=14, fontweight='bold')
    plt.ylabel('Pearson Correlation Coefficient')
    plt.xticks(rotation=45, ha='right')
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # Add correlation values on bars
    for i, (bar, corr) in enumerate(zip(bars, correlations)):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{corr:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print("Note: Blue bars indicate statistically significant correlations (p < 0.05)")
    print("Red bars indicate non-significant correlations (p >= 0.05)")

One can see that the correlations vary quite widely by industry. In the Manufacturing and Retail & Repair sections, the correlations are substantial, but not particularly strong. The correlation in section Other Community is the weakest of the 4 selected sections. On the other hand, in section "Hotels and Restaurants", the correlation is much higher than the other sections.

## Exploring the sources of financing for businesses

Now, let's explore some of the other characteristics of the business. The dataset contains information about sources of loans. The respondent is asked to give a series of yes/no answers about whether the business takes credit from various types of sources, such as banks or moneylenders. In order to create a plot that can allow us to compare the responses, the data requires some manipulation to achieve the right input format.

In [ ]:
# Create financing source analysis
# Map the financing source columns
finance_columns = {
    'q3_5a1': 'Private bank',
    'q3_5a2': 'Government bank', 
    'q3_5a3': 'Microfinance Organization',
    'q3_5a4': 'Moneylender',
    'q3_5a5': 'Family and Friends',
    'q3_5a6': 'Other'
}

# Create a long-format dataset for financing sources
finance_data_list = []

for col, source_name in finance_columns.items():
    if col in surveydata.columns:
        temp_df = surveydata[['slno', col, 'q4_7', 'ISIC_sectiondesc']].copy()
        temp_df.columns = ['slno', 'response', 'revenue', 'isic_section']
        temp_df['financesource'] = source_name
        temp_df['response'] = temp_df['response'].astype(str)
        finance_data_list.append(temp_df)

# Combine all financing source data
survey_finc_merged = pd.concat(finance_data_list, ignore_index=True)

# Convert response to proper format (1 = Yes, 2 = No typically)
survey_finc_merged['response_binary'] = survey_finc_merged['response'].map({'1': 'Yes', '2': 'No', '1.0': 'Yes', '2.0': 'No'})
survey_finc_merged = survey_finc_merged.dropna(subset=['response_binary'])

print(f"Finance data shape: {survey_finc_merged.shape}")
print(f"Financing sources: {survey_finc_merged['financesource'].unique()}")
print(f"Response distribution: {survey_finc_merged['response_binary'].value_counts()}")

In [ ]:
# Create faceted boxplots for financing sources
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Annual Revenue by Financing Source Usage', fontsize=16, fontweight='bold')

axes = axes.flatten()
finance_sources = survey_finc_merged['financesource'].unique()

for i, source in enumerate(finance_sources):
    if i < len(axes):
        source_data = survey_finc_merged[
            (survey_finc_merged['financesource'] == source) & 
            (survey_finc_merged['revenue'].notna())
        ]
        
        if len(source_data) > 0:
            sns.boxplot(data=source_data, x='response_binary', y='revenue', ax=axes[i])
            axes[i].set_ylim(0, 20000000)
            axes[i].set_title(f'{source}\n(n={len(source_data)})')
            axes[i].set_xlabel('Uses this financing source')
            axes[i].set_ylabel('Annual Revenue (Takas)' if i % 3 == 0 else '')
            axes[i].yaxis.set_major_formatter(formatter)
        else:
            axes[i].set_visible(False)

# Hide unused subplots
for j in range(len(finance_sources), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Summary statistics for firms that use each financing source
print("\n=== Summary Statistics for Firms Using Each Financing Source ===")
for source in finance_sources:
    yes_data = survey_finc_merged[
        (survey_finc_merged['financesource'] == source) & 
        (survey_finc_merged['response_binary'] == 'Yes') &
        (survey_finc_merged['revenue'].notna())
    ]['revenue']
    
    if len(yes_data) > 0:
        print(f"\n**{source}** (n={len(yes_data)}):")
        print(f"  Median: {yes_data.median():,.0f} Takas")
        print(f"  Mean: {yes_data.mean():,.0f} Takas")
        print(f"  Q1: {yes_data.quantile(0.25):,.0f} Takas")
        print(f"  Q3: {yes_data.quantile(0.75):,.0f} Takas")

These faceted boxplots reveal that the sources of credit (x axis, Yes/No) varied widely by annual revenue (y axis). Government and Private Banks showed by far a higher median revenue than the other options, and the difference between 1st and 3rd quartiles is much wider than the other sources, indicating that the Banks lent a wider variety of larger amounts. Firms with smaller revenue turned to Family and Friends, Moneylenders, and Microfinance organizations for credit; that they would do so is not surprising, considering that banks have traditionally lent to larger and more established firms.

## Does the lending vary by sector?

Let's look at how the different categories of lenders lend in different sectors.

In [ ]:
# Lending by sector analysis - focus on key financing sources
key_finance_sources = ['Government bank', 'Moneylender', 'Microfinance Organization']
key_sectors = ['Manufacturing', 'Retail & Repair', 'Hotels and restaurants', 'Logistics & Communications']

for finance_source in key_finance_sources:
    print(f"\n=== {finance_source.upper()} LENDING BY SECTOR ===")
    
    # Filter data for this financing source
    source_data = survey_finc_merged[
        (survey_finc_merged['financesource'] == finance_source) & 
        (survey_finc_merged['response_binary'] == 'Yes') &
        (survey_finc_merged['revenue'].notna()) &
        (survey_finc_merged['isic_section'].notna()) &
        (survey_finc_merged['isic_section'].isin(key_sectors))
    ]
    
    if len(source_data) > 0:
        # Create subplot
        plt.figure(figsize=(15, 8))
        
        # Box plot by sector
        sns.boxplot(data=source_data, x='isic_section', y='revenue', 
                   palette='Set2')
        
        plt.ylim(0, 30000000)
        plt.title(f'Annual Revenue for Firms Using {finance_source} by ISIC Section', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('ISIC Section')
        plt.ylabel('Annual Revenue (Takas)')
        plt.xticks(rotation=45, ha='right')
        plt.gca().yaxis.set_major_formatter(formatter)
        
        plt.tight_layout()
        plt.show()
        
        # Summary by sector
        sector_summary = source_data.groupby('isic_section')['revenue'].agg([
            'count', 'median', 'mean', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)
        ]).round(0)
        sector_summary.columns = ['Count', 'Median', 'Mean', 'Q1', 'Q3']
        print(sector_summary)
    else:
        print(f"No data available for {finance_source}")

# Final Plots and Summary

## Plot One

In [ ]:
# Final Plot 1: Enhanced geographic map with major cities
# Major cities data (manually entered based on the original R code)
major_cities = pd.DataFrame({
    'City': ['Dhaka', 'Chittagong', 'Khulna'],
    'Population': [10356500, 3390222, 1342339],
    'Longitude': [90.407, 91.832, 89.564],
    'Latitude': [23.71, 22.338, 22.81],
    'LatitudeName': [23.4, 22, 22.5]  # For label positioning
})

# Load geographic data again
districtmap = gpd.read_file('shp/bgd_polbnda_adm2_wfp.shp')
districtmap['ADM2_NAME_UPPER'] = districtmap['ADM2_NAME'].str.upper()

# Calculate averages by district
aggdata = surveydata.groupby('dist').agg({
    'q4_7': 'mean',
    'slno': 'count'
}).reset_index()
aggdata.columns = ['dist', 'avg_revenue', 'firm_count']
aggdata['dist_upper'] = aggdata['dist'].str.upper()

# Merge data
plot_data = districtmap.merge(aggdata, left_on='ADM2_NAME_UPPER', right_on='dist_upper', how='left')

# Create the enhanced map
fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Base map
districtmap.plot(ax=ax, color='lightgray', edgecolor='black', alpha=0.5, linewidth=0.5)

# Revenue data
plot_data_with_revenue = plot_data.dropna(subset=['avg_revenue'])
plot_data_with_revenue.plot(
    column='avg_revenue', 
    ax=ax, 
    legend=True,
    cmap='Blues',
    edgecolor='black',
    linewidth=0.5,
    legend_kwds={
        'label': 'Average Annual Revenue (Takas)', 
        'orientation': 'vertical', 
        'shrink': 0.6,
        'format': formatter
    }
)

# Add major cities
# Cities as circles scaled by population
for _, city in major_cities.iterrows():
    ax.scatter(city['Longitude'], city['Latitude'], 
              s=city['Population']/50000,  # Scale factor for visibility
              color='orange', alpha=0.7, edgecolor='red', linewidth=2)

# City labels
for _, city in major_cities.iterrows():
    ax.annotate(city['City'], 
               xy=(city['Longitude'], city['LatitudeName']),
               fontsize=12, fontweight='bold', color='red',
               ha='center')

ax.set_title('Figure 1: Average Annual Revenue by District with Major Cities\nBangladesh Informal Firms Survey 2010', 
            fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.grid(True, alpha=0.3)

# Add legend for city sizes
sizes = [1000000, 5000000, 10000000]
labels = ['1M', '5M', '10M']
legend_elements = []
for size, label in zip(sizes, labels):
    legend_elements.append(plt.scatter([], [], s=size/50000, color='orange', 
                                     alpha=0.7, edgecolor='red', linewidth=2, label=f'{label} people'))

ax.legend(handles=legend_elements, title='City Population', loc='upper right')

plt.tight_layout()
plt.show()

print("\n=== Geographic Analysis Summary ===")
print("Top 3 districts by average revenue:")
top_districts = aggdata.nlargest(3, 'avg_revenue')[['dist', 'avg_revenue', 'firm_count']]
for _, row in top_districts.iterrows():
    print(f"  {row['dist']}: {row['avg_revenue']:,.0f} Takas (n={row['firm_count']})")

### Description One

My first final plot is the map of Bangladesh that we used near the beginning of this project, adding some features to provide some context. First of all, I plotted on the map the 3 largest cities in Bangladesh, with orange circles indicating population size of each. As wealth often concentrates in urban areas (typically being centers of economic activity), I wanted to test the idea that firms with the largest level of revenue coincided with major urban centers in Bangladesh. In fact, in this case, we note that the highest average revenue coincides with the three major urban centers; Khulna district has the highest average revenue, and also contains the city of Khulna, the 3rd largest in Bangladesh. Chittagong district, with high average revenue, also contains the second-largest city in Bangladesh. And Dhaka district, where the capital and largest city of Bangladesh is located, also shows high revenue levels.

So, it seems that urbanization in Bangladesh coincides with high annual revenue for firms in those urban locations.

## Plot Two

In [ ]:
# Final Plot 2: Enhanced scatter plot with trend lines
selected_sections = ['Manufacturing', 'Retail & Repair', 'Hotels and restaurants']

# Filter data
plot2_data = surveydata[
    (surveydata['ISIC_sectiondesc'].isin(selected_sections)) &
    (surveydata['q4_7'].notna()) &
    (surveydata['q1_12a5a'].notna()) &
    (surveydata['q4_7'] > 0)  # For log transformation
]

# Create the plot
plt.figure(figsize=(15, 10))

# Color palette for sections
colors = {'Manufacturing': 'blue', 'Retail & Repair': 'red', 'Hotels and restaurants': 'green'}

# Plot points with jitter
for section in selected_sections:
    section_data = plot2_data[plot2_data['ISIC_sectiondesc'] == section]
    if len(section_data) > 0:
        # Add jitter to x-axis
        x_jitter = section_data['q1_12a5a'] + np.random.normal(0, 0.1, len(section_data))
        y_log = np.log10(section_data['q4_7'])
        
        plt.scatter(x_jitter, y_log, alpha=0.4, s=30, 
                   color=colors[section], label=section)

# Add trend lines
for section in selected_sections:
    section_data = plot2_data[plot2_data['ISIC_sectiondesc'] == section]
    if len(section_data) > 10:  # Minimum points for trend line
        x = section_data['q1_12a5a']
        y = np.log10(section_data['q4_7'])
        
        # Calculate trend line
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        
        # Plot trend line
        x_trend = np.linspace(x.min(), x.max(), 100)
        plt.plot(x_trend, p(x_trend), color=colors[section], linewidth=2, alpha=0.8)

plt.xlim(0, 20)
plt.xlabel('Number of Employees', fontsize=14)
plt.ylabel('Log10(Annual Revenue 2009, Takas)', fontsize=14)
plt.title('Size Characteristics of Firms in Bangladesh,\ngrouped by ISIC Section', 
         fontsize=16, fontweight='bold')
plt.legend(title='ISIC Section', fontsize=12, title_fontsize=12)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate trend line slopes
print("\n=== Trend Line Analysis ===")
for section in selected_sections:
    section_data = plot2_data[plot2_data['ISIC_sectiondesc'] == section]
    if len(section_data) > 10:
        x = section_data['q1_12a5a']
        y = np.log10(section_data['q4_7'])
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
        
        print(f"{section}:")
        print(f"  Slope: {slope:.4f} (log revenue increase per employee)")
        print(f"  R-squared: {r_value**2:.3f}")
        print(f"  P-value: {p_value:.6f}")

### Description Two

Plot two builds on the point plot comparison above, and refines to add trendlines to the plot. This plot has the following features:

* A log10 scale is applied to the Y axis (Annual Revenue 2009);
* Jitter is applied to avoid overclustering of points;
* The points are colored by ISIC section;
* Trendlines are added for select ISIC sections, to highlight the differences between them.

The trendlines emphasize the findings from the original plot: within the Retail & Repair section, annual revenue increases at a distinctly faster rate as employees increase, than in the Manufacturing and Hotels and restaurants sections. Using the combination of plotting methods, we have been able to identify a differentiating characteristic between certain sectors: at higher levels of revenue, Retail & Repair firms feature fewer employees than the Manufacturing and Hotels & Restaurants sections.

## Plot Three

In [ ]:
# Final Plot 3: Comprehensive financing analysis
# Select key financing sources and sectors
selected_finance = ['Microfinance Organization', 'Moneylender', 'Government bank']
selected_sectors = ['Manufacturing', 'Hotels and restaurants', 'Retail & Repair', 'Logistics & Communications']

# Filter and prepare data
plot3_data = survey_finc_merged[
    (survey_finc_merged['financesource'].isin(selected_finance)) &
    (survey_finc_merged['isic_section'].isin(selected_sectors)) &
    (survey_finc_merged['response_binary'] == 'Yes') &
    (survey_finc_merged['revenue'].notna())
]

# Set categorical order for better visualization
plot3_data['financesource'] = pd.Categorical(
    plot3_data['financesource'], 
    categories=['Microfinance Organization', 'Moneylender', 'Government bank']
)

plot3_data['isic_section'] = pd.Categorical(
    plot3_data['isic_section'],
    categories=['Manufacturing', 'Hotels and restaurants', 'Retail & Repair', 'Logistics & Communications']
)

# Create the plot
plt.figure(figsize=(16, 12))

# Create horizontal box plot with sectors as separate facets
g = sns.FacetGrid(plot3_data, row='isic_section', height=3, aspect=4, 
                  row_order=['Manufacturing', 'Hotels and restaurants', 'Retail & Repair', 'Logistics & Communications'])

g.map_dataframe(sns.boxplot, y='financesource', x='revenue', 
               palette='Set2', orient='h')

# Set consistent x-axis limits
for ax in g.axes.flat:
    ax.set_xlim(0, 45000000)
    ax.xaxis.set_major_formatter(formatter)
    ax.set_xlabel('Annual Revenue 2009 (Takas)')
    ax.set_ylabel('Source of Financing')

g.fig.suptitle('Annual Revenue 2009 for firms in Bangladesh,\nby ISIC Section and Sources of Financing', 
               fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()

# Summary statistics table
print("\n=== Summary: Median Revenue by Financing Source and Sector ===")
summary_table = plot3_data.groupby(['isic_section', 'financesource'])['revenue'].agg([
    'count', 'median', 'mean'
]).round(0)
summary_table.columns = ['Count', 'Median Revenue', 'Mean Revenue']
print(summary_table)

### Description Three

Plot three is a refinement of the box plots that were explored above. Those box plots informed the creation of plot three in a number of ways.

First, to make an effective and attractive visualization that captures the salient observations of our findings, it would be an idea to limit the number of items within the categories on the plot. For the ISIC section variable, I selected items that have enough observations to be able to draw meaningful insights; we can tell from the box plots and the point plots that items within the ISIC section category, such as "Health & Social Work" and "Other Community" do not have many observations--so it is difficult to glean any insights about those sections. I selected 3 sections: "Hotels and restaurants", "Manufacturing", and "Retail and Repair"--the items that I also highlighted in plot two with trend lines. Also, considering the other categorical variable, Source of Financing, I had noted that the 3 items from ISIC Section had displayed some distinctive characteristics as the source of financing change; additionally, I also included "Logistics and Communications", as for the Source of Financing "Microfinance Organization", the revenue patterns were distinctive when compared to the other ISIC sections.

Second, I noted that when faceting the Source of Financing "Government Banks", it was difficult to create a scale that could adequately show the range of responses for annual revenue; scaling on the y axis that appropriately showed the "Manufacturing" section would distort the plots of the other sections. To resolve this issue, I applied horizontal boxplots.

Third, I saw that being specific about the order of the data would help to bring out the story of the data. So, I adjusted the input data frame to order both categorical variables, to generally place the ISIC sections and sources of financing with higher annual revenue towards the top of the plot, while maintaining the order of sources of financing within each sub-plot to facilitate comparison.

This plot captures the relevant information that we gathered from the box plots. We can observe that government banks tend to provide credit to firms with higher levels of revenue, while moneylenders and microfinance organizations tend to lend to firms with relatively lower levels of revenue. We can also see the differentiated levels of revenue between the different ISIC sections, with Manufacturing firms having a markedly higher revenue than the other sectors.

Most interestingly, we can observe a potential segmentation of the finance market, by discerning patterns of how different lenders tend to lend to different industries. Microfinance organizations are designed to lend to firms of a very small size--and so one would expect that the firms who receive credit from such sources would have a lower annual revenue. This seems to hold true in most of the sections; however, we can see that firms in Logistics & communications that receive microfinance loans have a median revenue that is much higher than microfinance loan recipients in other sections. Government banks, on the other hand, may tend to target credit towards larger firms--hence the phenomenon of firms in the Manufacturing sector that take government bank loans, that show much higher levels of revenue than in other sections; hence, manufacturers seem to be a good market segment for government banks.

# Reflection

This process has been moderately successful in elucidating some insights about trends in Bangladeshi firms' annual revenue across a number of dimensions, such as geography, number of employees, industry sector, and financing source, with the intention of finding how these various characteristics affect and impact revenue, and more broadly, understand the dynamics of the market in which firms operate in Bangladesh.

My analysis came up with a number of interesting conclusions. First, we observed that annual revenue varied by geography, with the 3 districts having the highest average revenue also containing the 3 largest cities in Bangladesh. This fact supports the hypothesis that there is a positive correlation between urbanization and economic activity. For the purpose of this analysis, the ability to plot this information on a map proved to be essential.

We also have a better understanding of how the two measures of business size in this project (annual revenue and number of employees) were positively correlated, though not strongly so. Our analysis of this relationship was enhanced when we broke down the analysis using the ISIC section variable as an indicator of industry sector, to find that revenue in certain sectors (such as Retail and Repair) had a higher rate of growth as employees increased than in other sectors, such as Manufacturing. Box plots were particularly useful for exploring the ISIC section data; further, viewing the business size data on a point plot, colored by ISIC section, allowed us to see the differentiated trends of business size across industries.

Additionally, we have a better understanding of financing trends. Certain sources of finance, such as government banks, tend to lend to firms with higher revenue, while others, such as microfinance organizations, tend to focus on firms with low revenue. These trends in revenue may mean that sources of financing tend to target certain sectors; government banks lend to a large number of manufacturing firms, as those firms have a high level of revenue. Boxplots were used extensively to arrive at these conclusions.

One major limitation of this analysis was that such insights were limited to only a selection of ISIC sections; this limitation was probably due to the limited availability of observations for ISIC sections beyond those that we examined in depth.

One initial challenge that I had to work through was how to choose a limited set of variables to explore, when so many were available. This also means that there is a lot more data to explore within this data set; the data includes information such as family characteristics and education levels of business owners. Within this data set, I believe there are many other opportunities to segment businesses based on differing characteristics other than the ones we explored here.